In [1]:
import pandas as pd
from pathlib import Path
import gc # memory management

import optuna
from catboost import CatBoostClassifier, cv, Pool
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score

C:\miniforge3\envs\machine-learning\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## preprocessing


In [2]:
DATA_DIR = Path("../../data/kaggle-irrigation")
SEED = 42

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv", index_col="id")
train.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
id,,,,,,,,,,,,,,,,,,,,
0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [4]:
X = train.drop(columns="Irrigation_Need")
y = train["Irrigation_Need"]

# encode y
le = LabelEncoder()
y_encoded = le.fit_transform(y)

CAT_COLS = X.select_dtypes(include="object").columns.tolist()
cat_indices = [X.columns.tolist().index(c) for c in CAT_COLS]

C:\Users\Austin Muelrath\AppData\Local\Temp\ipykernel_15868\4134195525.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  CAT_COLS = X.select_dtypes(include="object").columns.tolist()


In [5]:
# split data
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.4, random_state=SEED, shuffle=True, stratify=y_encoded)

## base model

In [6]:
base_cb = CatBoostClassifier(random_state=SEED, cat_features=cat_indices, task_type="GPU")
base_cb.fit(X_train, y_train)

Learning rate set to 0.217817
0:	learn: 0.8554826	total: 20.5ms	remaining: 20.5s
1:	learn: 0.6071780	total: 32.5ms	remaining: 16.2s
2:	learn: 0.4598465	total: 42.8ms	remaining: 14.2s
3:	learn: 0.3609824	total: 53.5ms	remaining: 13.3s
4:	learn: 0.2906790	total: 63.6ms	remaining: 12.7s
5:	learn: 0.2390376	total: 75.5ms	remaining: 12.5s
6:	learn: 0.2004592	total: 86.4ms	remaining: 12.3s
7:	learn: 0.1711257	total: 97.1ms	remaining: 12s
8:	learn: 0.1490620	total: 109ms	remaining: 12s
9:	learn: 0.1314561	total: 119ms	remaining: 11.8s
10:	learn: 0.1183631	total: 130ms	remaining: 11.7s
11:	learn: 0.1074331	total: 141ms	remaining: 11.6s
12:	learn: 0.0996048	total: 151ms	remaining: 11.4s
13:	learn: 0.0930560	total: 160ms	remaining: 11.3s
14:	learn: 0.0875832	total: 171ms	remaining: 11.2s
15:	learn: 0.0831945	total: 180ms	remaining: 11.1s
16:	learn: 0.0797613	total: 189ms	remaining: 10.9s
17:	learn: 0.0771953	total: 199ms	remaining: 10.8s
18:	learn: 0.0748936	total: 209ms	remaining: 10.8s
19:	lea

In [7]:
y_preds_base = base_cb.predict(X_test)
balanced_accuracy_score(y_test, y_preds_base)

0.9605813522518641

## finetuning

In [8]:
def objective(trial):
    iterations = trial.suggest_int("iterations", 400, 1_000)
    learning_rate = trial.suggest_float("learning_rate", 0.02, 0.09)
    l2_leaf_reg = trial.suggest_float("l2_leaf_reg", 3, 4)
    bootstrap_type = trial.suggest_categorical("bootstrap_type", ["Bernoulli"])
    max_depth = trial.suggest_int("max_depth", 1, 16) # can only go up to 16 with gpu

    cb_optuna = CatBoostClassifier(
        random_seed=SEED,
        task_type="GPU",
        cat_features=cat_indices,
        iterations=iterations,
        learning_rate=learning_rate,
        l2_leaf_reg=l2_leaf_reg,
        bootstrap_type=bootstrap_type,
        max_depth=max_depth,
    )

    try:
        cb_optuna.fit(X_train, y_train)
        y_preds = cb_optuna.predict(X_test)
        score = balanced_accuracy_score(y_test, y_preds)
    finally:
        del cb_optuna
        gc.collect()

    return score

In [9]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=5)

[I 2026-04-16 17:37:09,048] A new study created in memory with name: no-name-13a67fef-6ec7-4d8a-b51b-7ccf1786fd9a


0:	learn: 1.0302626	total: 7.33ms	remaining: 3.37s
1:	learn: 0.9727181	total: 12.7ms	remaining: 2.91s
2:	learn: 0.9242421	total: 17.8ms	remaining: 2.71s
3:	learn: 0.8821757	total: 22.4ms	remaining: 2.56s
4:	learn: 0.8453204	total: 27.1ms	remaining: 2.47s
5:	learn: 0.8131637	total: 31.6ms	remaining: 2.39s
6:	learn: 0.7844608	total: 37.3ms	remaining: 2.41s
7:	learn: 0.7417585	total: 42.2ms	remaining: 2.38s
8:	learn: 0.7046870	total: 47ms	remaining: 2.35s
9:	learn: 0.6722373	total: 51.8ms	remaining: 2.33s
10:	learn: 0.6439020	total: 57.2ms	remaining: 2.33s
11:	learn: 0.6181560	total: 62.2ms	remaining: 2.32s
12:	learn: 0.5960448	total: 68.1ms	remaining: 2.34s
13:	learn: 0.5762307	total: 73.5ms	remaining: 2.34s
14:	learn: 0.5579186	total: 78.5ms	remaining: 2.33s
15:	learn: 0.5401247	total: 83.5ms	remaining: 2.32s
16:	learn: 0.5254411	total: 88.3ms	remaining: 2.3s
17:	learn: 0.5120648	total: 93.2ms	remaining: 2.29s
18:	learn: 0.4993215	total: 98.7ms	remaining: 2.29s
19:	learn: 0.4860104	tota

[I 2026-04-16 17:37:14,090] Trial 0 finished with value: 0.9579421778839013 and parameters: {'iterations': 460, 'learning_rate': 0.07595143939489676, 'l2_leaf_reg': 3.1113289161102196, 'bootstrap_type': 'Bernoulli', 'max_depth': 2}. Best is trial 0 with value: 0.9579421778839013.


0:	learn: 1.0818736	total: 6.66ms	remaining: 3.28s
1:	learn: 1.0659404	total: 12.4ms	remaining: 3.04s
2:	learn: 1.0507607	total: 17.9ms	remaining: 2.92s
3:	learn: 1.0362805	total: 23.4ms	remaining: 2.87s
4:	learn: 1.0224555	total: 29.6ms	remaining: 2.89s
5:	learn: 1.0092439	total: 34.9ms	remaining: 2.83s
6:	learn: 0.9966069	total: 40.4ms	remaining: 2.8s
7:	learn: 0.9845114	total: 45.7ms	remaining: 2.77s
8:	learn: 0.9729230	total: 51.7ms	remaining: 2.78s
9:	learn: 0.9618124	total: 57.4ms	remaining: 2.77s
10:	learn: 0.9511558	total: 64.3ms	remaining: 2.82s
11:	learn: 0.9409249	total: 69.8ms	remaining: 2.8s
12:	learn: 0.9310997	total: 77.1ms	remaining: 2.85s
13:	learn: 0.9216552	total: 81.6ms	remaining: 2.79s
14:	learn: 0.9125751	total: 88.6ms	remaining: 2.82s
15:	learn: 0.9038376	total: 94.5ms	remaining: 2.82s
16:	learn: 0.8954270	total: 100ms	remaining: 2.81s
17:	learn: 0.8873275	total: 107ms	remaining: 2.82s
18:	learn: 0.8795250	total: 113ms	remaining: 2.81s
19:	learn: 0.8720024	total:

[I 2026-04-16 17:37:19,414] Trial 1 finished with value: 0.7029220462986405 and parameters: {'iterations': 493, 'learning_rate': 0.02028575941899529, 'l2_leaf_reg': 3.702797715892871, 'bootstrap_type': 'Bernoulli', 'max_depth': 1}. Best is trial 0 with value: 0.9579421778839013.


...

[I 2026-04-16 17:37:53,520] Trial 2 finished with value: 0.9588830362022006 and parameters: {'iterations': 984, 'learning_rate': 0.03877222174318945, 'l2_leaf_reg': 3.7335578701850767, 'bootstrap_type': 'Bernoulli', 'max_depth': 10}. Best is trial 2 with value: 0.9588830362022006.


0:	learn: 1.0019559	total: 16.9ms	remaining: 9.9s
1:	learn: 0.9239830	total: 32.3ms	remaining: 9.45s
2:	learn: 0.8137886	total: 48.1ms	remaining: 9.37s
3:	learn: 0.7236749	total: 63.7ms	remaining: 9.28s
4:	learn: 0.6484466	total: 79.4ms	remaining: 9.24s
5:	learn: 0.5845236	total: 94.8ms	remaining: 9.18s
6:	learn: 0.5296121	total: 110ms	remaining: 9.13s
7:	learn: 0.4819288	total: 128ms	remaining: 9.26s
8:	learn: 0.4401904	total: 145ms	remaining: 9.29s
9:	learn: 0.4034460	total: 160ms	remaining: 9.22s
10:	learn: 0.3709543	total: 176ms	remaining: 9.21s
11:	learn: 0.3420493	total: 192ms	remaining: 9.21s
12:	learn: 0.3162769	total: 208ms	remaining: 9.19s
13:	learn: 0.2931920	total: 224ms	remaining: 9.18s
14:	learn: 0.2724831	total: 245ms	remaining: 9.33s
15:	learn: 0.2538230	total: 263ms	remaining: 9.38s
16:	learn: 0.2370004	total: 281ms	remaining: 9.42s
17:	learn: 0.2218368	total: 299ms	remaining: 9.45s
18:	learn: 0.2080787	total: 314ms	remaining: 9.39s
19:	learn: 0.1956741	total: 329ms	re

[I 2026-04-16 17:38:05,311] Trial 3 finished with value: 0.9591528912984809 and parameters: {'iterations': 587, 'learning_rate': 0.07983556916867832, 'l2_leaf_reg': 3.8726669656276553, 'bootstrap_type': 'Bernoulli', 'max_depth': 7}. Best is trial 3 with value: 0.9591528912984809.


0:	learn: 1.0191591	total: 24.9ms	remaining: 16.5s
1:	learn: 0.9524917	total: 48.8ms	remaining: 16.2s
2:	learn: 0.8953178	total: 72ms	remaining: 15.9s
3:	learn: 0.8088802	total: 97.7ms	remaining: 16.2s
4:	learn: 0.7353051	total: 124ms	remaining: 16.4s
5:	learn: 0.6717737	total: 148ms	remaining: 16.3s
6:	learn: 0.6163055	total: 172ms	remaining: 16.2s
7:	learn: 0.5674488	total: 198ms	remaining: 16.3s
8:	learn: 0.5241089	total: 223ms	remaining: 16.3s
9:	learn: 0.4854343	total: 247ms	remaining: 16.2s
10:	learn: 0.4507135	total: 271ms	remaining: 16.1s
11:	learn: 0.4194702	total: 296ms	remaining: 16.1s
12:	learn: 0.3911937	total: 320ms	remaining: 16.1s
13:	learn: 0.3655554	total: 345ms	remaining: 16.1s
14:	learn: 0.3422634	total: 366ms	remaining: 15.9s
15:	learn: 0.3210083	total: 386ms	remaining: 15.7s
16:	learn: 0.3016166	total: 412ms	remaining: 15.7s
17:	learn: 0.2838432	total: 437ms	remaining: 15.7s
18:	learn: 0.2675525	total: 460ms	remaining: 15.6s
19:	learn: 0.2526158	total: 481ms	remai

[I 2026-04-16 17:38:27,232] Trial 4 finished with value: 0.9598352447950292 and parameters: {'iterations': 666, 'learning_rate': 0.06501407166933859, 'l2_leaf_reg': 3.3708479680987855, 'bootstrap_type': 'Bernoulli', 'max_depth': 8}. Best is trial 4 with value: 0.9598352447950292.


In [11]:
best_cb = CatBoostClassifier(
    random_seed=SEED,
    task_type="GPU",
    cat_features=cat_indices,
    **study.best_params
)

best_cb.fit(X_train, y_train)
y_preds = best_cb.predict(X_test)

balanced_accuracy_score(y_test, y_preds)

0:	learn: 1.0191591	total: 21.8ms	remaining: 14.5s
1:	learn: 0.9524918	total: 42.1ms	remaining: 14s
2:	learn: 0.8953177	total: 61.1ms	remaining: 13.5s
3:	learn: 0.8088801	total: 81.3ms	remaining: 13.5s
4:	learn: 0.7353051	total: 106ms	remaining: 14s
5:	learn: 0.6717737	total: 133ms	remaining: 14.6s
6:	learn: 0.6163055	total: 159ms	remaining: 15s
7:	learn: 0.5674489	total: 184ms	remaining: 15.1s
8:	learn: 0.5241088	total: 208ms	remaining: 15.2s
9:	learn: 0.4854344	total: 236ms	remaining: 15.4s
10:	learn: 0.4507135	total: 261ms	remaining: 15.5s
11:	learn: 0.4194702	total: 284ms	remaining: 15.5s
12:	learn: 0.3911937	total: 302ms	remaining: 15.2s
13:	learn: 0.3655553	total: 326ms	remaining: 15.2s
14:	learn: 0.3422633	total: 345ms	remaining: 15s
15:	learn: 0.3210083	total: 365ms	remaining: 14.8s
16:	learn: 0.3016166	total: 383ms	remaining: 14.6s
17:	learn: 0.2838432	total: 403ms	remaining: 14.5s
18:	learn: 0.2675525	total: 420ms	remaining: 14.3s
19:	learn: 0.2526157	total: 438ms	remaining: 

0.9596613957773411

## submission

In [12]:
test = pd.read_csv(DATA_DIR / "test.csv", index_col="id")
test.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
id,,,,,,,,,,,,,,,,,,,
630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central


In [15]:
final_preds = pd.DataFrame({"Irrigation_Need": le.inverse_transform(best_cb.predict(test))}, index=test.index)
final_preds.index.name = "id"

final_preds

C:\miniforge3\envs\machine-learning\Lib\site-packages\sklearn\preprocessing\_label.py:161: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


,Irrigation_Need
id,
630000,Low
630001,Low
630002,Low
630003,Low
630004,Low
...,...
899995,Medium
899996,Low
899997,Medium


In [16]:
final_preds.to_csv(DATA_DIR / "boosting_preds.csv")